# FinExplain: QLoRA Fine-Tuning Qwen2.5-1.5B-Instruct on FinQA

In [ ]:
!pip install -q -U torch transformers peft bitsandbytes datasets accelerate pypdf rank-bm25

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
from datasets import load_dataset
dataset = load_dataset('financial_phrasebank', 'sentences_allagree', split='train[:500]')
print(f'Loaded dataset sample size: {len(dataset)}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import shutil
OUTPUT_DIR = './finexplain_qwen_lora'
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
shutil.make_archive('finexplain_qwen_lora', 'zip', OUTPUT_DIR)
print('Adapter ZIP created successfully!')